# RBM+TDVP vs exact ED at $\theta = 0.45\pi$

Evaluates the `rbm_tdvp_scan_theta0.45pi_fine.npz` run (produced by `rbm_tdvp_quench_scan.py --theta 1.4137166941 --tag theta0.45pi_fine --n_iter 2000 --ngf 30`)
the same way `rbm_vs_ed_comparison.ipynb` evaluates every run: exact-ED cluster-expansion
Fermi-Dirac fit ($\mu$, $T$, $R^2$ of rank vs $\log|c_A|$) on a $(t, g_f)$ grid, alongside the
RBM+TDVP infidelity vs exact.

Motivation ([note.md](note.md)): a hypothesis that at $\theta=0.45\pi$, $\mu$ is always large
immediately after the quench and that the early-time dynamics are roughly independent of $g_f$.
The heatmaps below, plus the explicit $\mu(g_f)$ cuts at fixed times, test that directly.

In [ ]:
import os
import sys
sys.path.append('../ClusterExpansion/')
sys.path.append('.')

import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import LogNorm

import cluster_expansion as ce
import ed_definitions as ed

## Shared Fermi-Dirac fit machinery (same as `rbm_vs_ed_comparison.ipynb`)

In [ ]:
def fermi_dirac(x, mu, T):
    return 1024 / (np.exp((x - mu) / T) + 1)

noise_floor = -28.0  # log|c_A| below this is Z2-symmetry-forced machine-precision noise

def rank_vs_logc(c_A_row):
    """Sort |c_A| descending, drop numerical-noise entries, return (log|c_A|, rank)."""
    absvals = np.abs(c_A_row)
    with np.errstate(divide='ignore'):
        log_c_all = np.log(absvals)
    real_sorted = np.sort(absvals[log_c_all > noise_floor])[::-1]
    log_c = np.log(real_sorted)
    rank = np.arange(1, len(real_sorted) + 1)
    return log_c, rank

def compute_ed_grid(L, J, g_i, J2, gz, theta, pbc, g_f_values, times):
    """Exact-ED mu(t,g_f), T(t,g_f), R^2(t,g_f) for the given quench family."""
    n_times = len(times)
    _, hi, H_i = ed.build_TFI_hamiltonian_2(L, g_i, J, J2, gz=gz, theta=theta, pbc=pbc)
    psi0 = ed.ground_state(H_i)[1]

    mu_grid = np.full((len(g_f_values), n_times), np.nan)
    T_grid = np.full((len(g_f_values), n_times), np.nan)
    R2_grid = np.full((len(g_f_values), n_times), np.nan)

    for i, g_f in enumerate(g_f_values):
        _, _, H_f = ed.build_TFI_hamiltonian_2(L, g_f, J, J2, gz=gz, theta=theta, pbc=pbc)
        evals, evecs = ed.full_diagonalization(H_f)
        psi_t = ed.evolve_state(evals, evecs, psi0, times)
        p0 = [-10.0, 3.0]
        for k in range(n_times):
            logabs = np.log(np.abs(psi_t[k]))
            c_A_row = np.real(ce.fwht_coeffs_in_cluster_col_order(logabs, hi))
            log_c, rank = rank_vs_logc(c_A_row)
            try:
                popt, _ = curve_fit(fermi_dirac, log_c, rank, p0=p0, maxfev=20000)
                mu_grid[i, k] = popt[0]
                T_grid[i, k] = popt[1]
                pred = fermi_dirac(log_c, *popt)
                R2_grid[i, k] = 1 - np.sum((rank - pred) ** 2) / np.sum((rank - rank.mean()) ** 2)
                p0 = popt
            except RuntimeError:
                pass
    return mu_grid, T_grid, R2_grid

def load_or_compute_ed_grid(fpath, ed_ce_dir, L, J, g_i, J2, gz, theta, pbc, g_f_values, times):
    """Load the precomputed ED grid for this run from ED_CE/ if it exists,
    otherwise compute it and cache it there for next time."""
    cache_path = f'{ed_ce_dir}/{os.path.basename(fpath)}'
    if os.path.exists(cache_path):
        c = np.load(cache_path)
        return c['mu_grid'], c['T_grid'], c['R2_grid']

    mu_grid, T_grid, R2_grid = compute_ed_grid(L, J, g_i, J2, gz, theta, pbc, g_f_values, times)
    os.makedirs(ed_ce_dir, exist_ok=True)
    np.savez(cache_path, mu_grid=mu_grid, T_grid=T_grid, R2_grid=R2_grid,
             g_f_values=g_f_values, times=times, L=L, J=J, g_i=g_i, J2=J2, gz=gz, theta=theta, pbc=pbc)
    return mu_grid, T_grid, R2_grid

## Load the $\theta=0.45\pi$ run and compute/cache the matching ED grid

In [ ]:
data_dir = '../data/cumulants_Fermi_dirac_TDVP'
ed_ce_dir = f'{data_dir}/ED_CE'
fpath = f'{data_dir}/rbm_tdvp_scan_theta0.45pi_fine.npz'

d = np.load(fpath)
L = int(d['L']); J = float(d['J']); g_i = float(d['g_i']); pbc = bool(d['pbc'])
alpha = int(d['alpha'])
J2 = float(d['J2']) if 'J2' in d else 0.0
gz = float(d['gz']) if 'gz' in d else 0.0
theta = float(d['theta']) if 'theta' in d else 0.0
g_f_values = d['g_f_values']
times = d['times']
infid_grid = d['infidelity_vs_exact']
init_infid = float(d['initial_infidelity'])

print(f"L={L} J={J} J2={J2} gz={gz} theta={theta:.4f} (={theta/np.pi:.3f} pi) g_i={g_i} pbc={pbc} alpha={alpha}")
print(f"g_f grid: {len(g_f_values)} points in [{g_f_values.min():.2f}, {g_f_values.max():.2f}]")
print(f"times: {len(times)} points in [{times.min():.2f}, {times.max():.2f}]")
print(f"initial infidelity (RBM vs exact g_i ground state): {init_infid:.3e}")

mu_grid, T_grid, R2_grid = load_or_compute_ed_grid(
    fpath, ed_ce_dir, L, J, g_i, J2, gz, theta, pbc, g_f_values, times)

## Heatmaps: infidelity, $\mu$, $T$, $R^2$ over $(t, g_f)$

In [ ]:
CUT_POSITIONS = [0.3, 0.7, 1.5, 2.2, 2.8]   # g_f values to cut, spanning below/at/above g_i
CUT_COLORS = ['tab:blue', 'tab:green', 'black', 'tab:orange', 'tab:red']
cut_idx = [int(np.argmin(np.abs(g_f_values - pos))) for pos in CUT_POSITIONS]

col_titles = ['infidelity $(t, g_f)$', r'$\mu(t, g_f)$', r'$T(t, g_f)$', r'$R^2(t, g_f)$']
cbar_labels = ['infidelity', r'$\mu$', r'$T$', r'$R^2$']
grids = [infid_grid, mu_grid, T_grid, R2_grid]
kwargs = [
    dict(norm=LogNorm(vmin=1e-6, vmax=max(1e-2, np.nanmax(infid_grid)))),
    {},
    {},
    dict(vmin=0.9, vmax=1.0),
]

outline = [pe.withStroke(linewidth=1.0, foreground='black')]
theta_math = f"{theta/np.pi:.2f}\\pi" if theta != 0 else "0"
param_str = (rf"$J={J:g}$" + "\n" + rf"$J_2={J2:g}$" + "\n" + rf"$g^z={gz:g}$" + "\n"
             + rf"$\theta={theta_math}$" + "\n" + rf"$g_i={g_i:g}$")

fig, axs = plt.subplots(1, 4, figsize=(20, 4), constrained_layout=True)
for col, (grid, kw) in enumerate(zip(grids, kwargs)):
    im = axs[col].pcolormesh(times, g_f_values, grid, shading='auto', cmap='viridis', **kw)
    axs[col].axhline(g_i, color='white', ls='-', lw=1)
    for idx, c in zip(cut_idx, CUT_COLORS):
        axs[col].axhline(g_f_values[idx], color=c, ls='--', lw=1.2)
    axs[col].set_title(col_titles[col])
    axs[col].set_xlabel('$t$')
    fig.colorbar(im, ax=axs[col], label=cbar_labels[col])
axs[0].set_ylabel('$g_f$')
axs[0].text(0.03, 0.97, param_str, transform=axs[0].transAxes, ha='left', va='top',
            fontsize=11, color='white', path_effects=outline)
plt.show()

## Cuts at fixed $g_f$: is the early-time behavior really independent of $g_f$?

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(20, 3.5), constrained_layout=True)
for idx, c in zip(cut_idx, CUT_COLORS):
    gf_actual = g_f_values[idx]
    label = rf"$g_f={gf_actual:.2f}$"
    axs[0].plot(times, infid_grid[idx], color=c, label=label)
    axs[1].plot(times, mu_grid[idx], color=c, label=label)
    axs[2].plot(times, T_grid[idx], color=c, label=label)
    axs[3].plot(times, R2_grid[idx], color=c, label=label)
axs[0].set_yscale('log')
axs[0].set_title('infidelity$(t)$'); axs[1].set_title(r'$\mu(t)$')
axs[2].set_title(r'$T(t)$'); axs[3].set_title(r'$R^2(t)$')
for ax in axs:
    ax.set_xlabel('$t$')
    ax.grid(alpha=0.3)
axs[1].legend(fontsize=8)
plt.show()

## $\mu$ and $T$ vs $g_f$ at fixed times

Directly tests the note.md hypothesis: if the early-time dynamics at $\theta=0.45\pi$ are
independent of $g_f$, the curves for the earliest times should be roughly flat in $g_f$.

In [ ]:
TIME_SNAPSHOTS = [0.0, 0.05, 0.1, 0.2, 0.4, times.max()]
time_idx = [int(np.argmin(np.abs(times - t))) for t in TIME_SNAPSHOTS]
cmap = plt.get_cmap('viridis')

fig, axs = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for n, k in enumerate(time_idx):
    color = cmap(n / max(1, len(time_idx) - 1))
    t_actual = times[k]
    axs[0].plot(g_f_values, mu_grid[:, k], color=color, marker='o', ms=3, label=f'$t={t_actual:.2f}$')
    axs[1].plot(g_f_values, T_grid[:, k], color=color, marker='o', ms=3, label=f'$t={t_actual:.2f}$')
for ax in axs:
    ax.axvline(g_i, color='gray', ls=':', lw=1, label='$g_i$' if ax is axs[0] else None)
    ax.set_xlabel('$g_f$')
    ax.grid(alpha=0.3)
axs[0].set_ylabel(r'$\mu$'); axs[0].set_title(r'$\mu(g_f)$ at fixed $t$')
axs[1].set_ylabel(r'$T$'); axs[1].set_title(r'$T(g_f)$ at fixed $t$')
axs[0].legend(fontsize=8)
plt.show()